In [2]:
import urllib.request
import ssl
import zipfile
import os
from pathlib import Path

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path = "sms_spam_collection.zip"
extracted_path = "sms_spam_collection"
data_file_path = Path(extracted_path) / "SMSSpamCollection.tsv"

def download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path):
    if data_file_path.exists():
        print(f"{data_file_path} already exists. Skipping download and extraction.")
        return

    # Create an unverified SSL context
    ssl_context = ssl._create_unverified_context()

    # Downloading the file
    with urllib.request.urlopen(url, context=ssl_context) as response:
        with open(zip_path, "wb") as out_file:
            out_file.write(response.read())

    # Unzipping the file
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extracted_path)

    # Add .tsv file extension
    original_file_path = Path(extracted_path) / "SMSSpamCollection"
    os.rename(original_file_path, data_file_path)
    print(f"File downloaded and saved as {data_file_path}")

download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)

File downloaded and saved as sms_spam_collection\SMSSpamCollection.tsv


In [4]:
import pandas as pd

In [5]:
df=pd.read_csv(data_file_path, sep="\t", header=None, names=["label", "text"])

In [10]:
df['label'].value_counts()

label
ham     4825
spam     747
Name: count, dtype: int64

In [11]:
n = df['label'].value_counts().min()  # number of samples in the minority class

spam_df = df[df["label"] == "spam"].sample(n=n, random_state=42)
ham_df  = df[df["label"] == "ham"].sample(n=n, random_state=42)

balanced_df = pd.concat([spam_df, ham_df], ignore_index=True)
balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle

print(balanced_df["label"].value_counts())

label
ham     747
spam    747
Name: count, dtype: int64


In [12]:
balanced_df["label"] = balanced_df["label"].map({"ham": 0, "spam": 1})

In [14]:
def random_split(df, train_frac=0.8, val_frac=0.1, test_frac=0.1, random_state=42):
    assert train_frac + val_frac + test_frac == 1, "Fractions must sum to 1"
    
    df = df.sample(frac=1, random_state=random_state).reset_index(drop=True)  # Shuffle the DataFrame
    
    n = len(df)
    train_end = int(train_frac * n)
    val_end = train_end + int(val_frac * n)
    
    train_df = df.iloc[:train_end]
    val_df = df.iloc[train_end:val_end]
    test_df = df.iloc[val_end:]
    
    return train_df, val_df, test_df

train_df, val_df, test_df = random_split(balanced_df,train_frac=0.7, val_frac=0.1, test_frac=0.2, random_state=42)

In [15]:
len(train_df), len(val_df), len(test_df)

(1045, 149, 300)

In [16]:
train_df.to_csv("train.csv", index=False)
val_df.to_csv("val.csv", index=False)   
test_df.to_csv("test.csv", index=False)

In [ ]:
max_len = balanced_df.astype(str).applymap(len).max().max()
max_len

C:\Users\rouna\AppData\Local\Temp\ipykernel_5836\3442238167.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  max_len = balanced_df.astype(str).applymap(len).max().max()


np.int64(444)

In [19]:
len(train_df['text'][0])

39